# Regression vs Classification Selection

**Topic:** Supervised Learning — Problem Framing

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import RadioButtons, Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, roc_auc_score
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the criteria that determine whether a problem should be framed as regression or classification
- **Explain** when the same dataset can legitimately be approached as either, and what changes
- **Interpret** a side-by-side comparison showing how the two framings answer different questions

> **Tip:** In the decision framework widget, answer each question about your target variable and constraints. Notice that the boundary between regression and classification is often a business decision, not a data decision.

---
## How we got here

This notebook brings together the regression and classification families you have studied:

- **[supervised/00_supervised_learning_overview.ipynb](00_supervised_learning_overview.ipynb)** — introduced the regression vs classification split; this notebook goes deeper on how to make the choice deliberately
- **[supervised/01_linear_regression.ipynb](01_linear_regression.ipynb)** through **[supervised/04_ridge_lasso_regression.ipynb](04_ridge_lasso_regression.ipynb)** — regression algorithms you can choose when the framing fits
- **[supervised/05_logistic_regression.ipynb](05_logistic_regression.ipynb)** through **[supervised/12_naive_bayes.ipynb](12_naive_bayes.ipynb)** — classification algorithms you can choose when the framing fits

---
## Why this matters for data science

Choosing the wrong problem framing is one of the most common and most expensive mistakes in applied machine learning. You can have the perfect model, perfect data, and perfect code — and still deliver the wrong answer if you framed the problem incorrectly.

For example: predicting whether a customer will spend more than $100 this month is classification. Predicting how much they will spend is regression. Both are valid, but they answer different questions and require different evaluation metrics. Choosing framing based on what the business actually needs, not what is easiest to model, is a core data science skill.

In [ ]:
target_rb = widgets.RadioButtons(
    options=["Continuous number", "Ordered category", "Unordered category", "Binary yes/no"],
    description="1. Target variable is:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="420px"),
)
need_rb = widgets.RadioButtons(
    options=["Exact prediction", "Probability", "Category label", "Ranking"],
    description="2. Business needs:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="420px"),
)
interp_rb = widgets.RadioButtons(
    options=["Must explain every prediction", "Broad trends are fine", "Black box is OK"],
    description="3. Interpretability need:",
    style={"description_width": "200px"},
    layout=widgets.Layout(width="420px"),
)

out = Output()

# Weighted votes toward Regression vs Classification for each answer.
# Ties (e.g. "Ordered category") lean toward the more common real-world choice
# but the framing widget will call out genuine ambiguity rather than force a pick.
TARGET_MAP = {
    "Continuous number":  ("Regression", 2),
    "Ordered category":   ("Classification", 1),
    "Unordered category": ("Classification", 2),
    "Binary yes/no":      ("Classification", 2),
}
NEED_MAP = {
    "Exact prediction": ("Regression", 2),
    "Probability":      ("Classification", 1),
    "Category label":   ("Classification", 2),
    "Ranking":          ("Regression", 1),
}
INTERP_MAP = {
    "Must explain every prediction": "Linear/Logistic Regression, or a shallow Decision Tree",
    "Broad trends are fine":         "Random Forest or Gradient Boosting",
    "Black box is OK":               "Gradient Boosting, XGBoost, or a Neural Network",
}

# Tree layout — a fixed data-coordinate canvas the (hidden) axes are pinned to.
ROOT_XY, ROOT_WH = (6, 7.5), (3.6, 1.1)
REG_XY, CLF_XY, BRANCH_WH = (3, 5), (9, 5), (3.6, 1.3)
LEAF_Y, LEAF_WH = 2.3, (3.6, 1.3)
LOSE_OPACITY = 0.4

def wrap_label(text, threshold=28):
    """Break a long label onto two lines at the comma/space nearest its midpoint."""
    if len(text) <= threshold:
        return text
    breakpoints = [i for i, ch in enumerate(text) if ch in ", "]
    if not breakpoints:
        return text
    mid = len(text) / 2
    best = min(breakpoints, key=lambda i: abs(i - mid))
    return text[:best + 1].rstrip() + "<br>" + text[best + 1:].lstrip()

def render(change=None):
    reg_score, clf_score = 0, 0
    for mapping, choice in ((TARGET_MAP, target_rb.value), (NEED_MAP, need_rb.value)):
        weight_framing, weight = mapping[choice]
        if weight_framing == "Regression":
            reg_score += weight
        else:
            clf_score += weight

    if reg_score > clf_score:
        framing = "Regression"
    elif clf_score > reg_score:
        framing = "Classification"
    else:
        framing = "Either — genuinely ambiguous, let the business need decide"

    state = {"Regression": "regression", "Classification": "classification"}.get(framing, "tie")
    algo_family = INTERP_MAP[interp_rb.value]

    win_color, lose_color, root_color = PALETTE["primary"], PALETTE["muted"], PALETTE["muted"]

    fig = go.Figure(layout=base_layout(title="", xaxis_title="", yaxis_title=""))
    fig.add_trace(go.Scatter(x=[0, 12], y=[0, 9], mode="markers",
                              marker=dict(opacity=0), hoverinfo="skip", showlegend=False))

    def add_box(cx, cy, w, h, title, subtitle, color, opacity, text_color):
        fig.add_shape(type="rect", x0=cx - w / 2, x1=cx + w / 2, y0=cy - h / 2, y1=cy + h / 2,
                      line=dict(color=color, width=2), fillcolor=color, opacity=opacity)
        fig.add_annotation(x=cx, y=cy, text=f"<b>{title}</b><br>{subtitle}",
                            showarrow=False, align="center", font=dict(size=13, color=text_color))

    def add_arrow(x0, y0, x1, y1, color, opacity=1.0):
        fig.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref="x", yref="y", axref="x", ayref="y",
                            text="", showarrow=True, arrowhead=2, arrowsize=1.2,
                            arrowwidth=2, arrowcolor=color, opacity=opacity)

    root_bottom = ROOT_XY[1] - ROOT_WH[1] / 2
    branch_top = REG_XY[1] + BRANCH_WH[1] / 2
    branch_bottom = REG_XY[1] - BRANCH_WH[1] / 2
    leaf_top = LEAF_Y + LEAF_WH[1] / 2

    add_box(*ROOT_XY, *ROOT_WH, "Your answers", "3 questions answered", root_color, 1.0, "white")

    reg_active = state in ("regression", "tie")
    clf_active = state in ("classification", "tie")

    add_box(*REG_XY, *BRANCH_WH, "Regression", f"Signal {reg_score} vs {clf_score}",
            win_color if reg_active else lose_color,
            1.0 if reg_active else LOSE_OPACITY,
            "white" if reg_active else lose_color)
    add_box(*CLF_XY, *BRANCH_WH, "Classification", f"Signal {clf_score} vs {reg_score}",
            win_color if clf_active else lose_color,
            1.0 if clf_active else LOSE_OPACITY,
            "white" if clf_active else lose_color)

    if state == "regression":
        leaf_x = REG_XY[0]
        add_arrow(ROOT_XY[0], root_bottom, REG_XY[0], branch_top, win_color)
        add_arrow(REG_XY[0], branch_bottom, leaf_x, leaf_top, win_color)
    elif state == "classification":
        leaf_x = CLF_XY[0]
        add_arrow(ROOT_XY[0], root_bottom, CLF_XY[0], branch_top, win_color)
        add_arrow(CLF_XY[0], branch_bottom, leaf_x, leaf_top, win_color)
    else:
        leaf_x = 6
        add_arrow(ROOT_XY[0], root_bottom, REG_XY[0], branch_top, win_color)
        add_arrow(ROOT_XY[0], root_bottom, CLF_XY[0], branch_top, win_color)
        add_arrow(REG_XY[0], branch_bottom, leaf_x, leaf_top, win_color)
        add_arrow(CLF_XY[0], branch_bottom, leaf_x, leaf_top, win_color)

    leaf_subtitle = "Either framing works" if state == "tie" else wrap_label(algo_family)
    add_box(leaf_x, LEAF_Y, *LEAF_WH, "Recommended", leaf_subtitle, win_color, 1.0, "white")

    fig.update_layout(
        height=440, showlegend=False,
        xaxis=dict(visible=False, range=[0, 12], fixedrange=True, showgrid=False, zeroline=False),
        yaxis=dict(visible=False, range=[0, 9], fixedrange=True, showgrid=False, zeroline=False),
        margin=dict(l=20, r=20, t=20, b=20),
    )

    with out:
        clear_output(wait=True)
        display(widgets.HTML(
            f"<b>Recommended framing:</b> {framing}<br>"
            f"<b>Algorithm family to start with:</b> {algo_family}"
        ))
        fig.show(config={"displayModeBar": False})

target_rb.observe(render, names="value")
need_rb.observe(render, names="value")
interp_rb.observe(render, names="value")

display(VBox([
    HBox([target_rb, need_rb, interp_rb]),
    out,
]))
render()

---
## What's happening?

The same dataset can often be framed as either regression or classification, and the choice should be driven by what the model's output will be used for.

**Frame as regression when** you need the actual magnitude: how many days until a machine fails, what will the patient's blood pressure be next month, what price will this item sell for.

**Frame as classification when** you need a decision: will this customer churn (yes/no), is this transaction fraudulent (yes/no), which of three diagnoses fits best.

**Ambiguous cases** where either is valid:
- Predicting hospital readmission: "within 30 days" is classification; "days until readmission" is regression
- Sentiment analysis: "positive/negative/neutral" is classification; "sentiment score 0-100" is regression
- Credit risk: "default/no default" is classification; "probability of default" is regression — but the latter uses a classifier's output, not a regressor

The framing choice also determines your evaluation metric. Classification uses accuracy, F1, AUC. Regression uses RMSE, MAE, R².

---
## Real-world example: Same data, two framings

The California Housing dataset can be framed as a regression problem (predict exact price) or a classification problem (predict whether a district is "high value": above median price). Both use the same features and training data.

- **Notice:** The regression model returns a price estimate; the classification model returns a probability of being high-value — these are different kinds of answers
- **Notice:** R² tells you how much of the price variance the regression model explains; AUC tells you how well the classifier ranks high-value vs low-value districts
- **Notice:** You cannot compare R² and AUC directly — they measure different things about different outputs from different framings of the same data

> **Discussion question:** A real estate agent wants to flag listings that are "likely underpriced." Should they use regression (predict price, compare to listing) or classification (predict underpriced/fair/overpriced)? What additional information do you need to answer this?

### Decision guide: regression or classification?

| Scenario | Target variable | Framing | Why |
|---|---|---|---|
| Predict customer spend | Amount in dollars | Regression | Need the exact number for budgeting |
| Flag likely churners | Churn: yes/no | Classification | Need a list of at-risk customers |
| Score lead quality | Probability of converting | Classification | Probability is a score used for ranking |
| Predict loan default | Default: yes/no | Classification | Binary outcome, regulatory threshold needed |
| Predict patient recovery time | Days until discharge | Regression | Staffing decisions need the actual number |
| Detect anomalous transactions | Anomaly: yes/no | Classification | Binary alert system, human reviews flags |

In [ ]:
from plotly.subplots import make_subplots

np.random.seed(42)
housing = fetch_california_housing(as_frame=True)
X, y_cont = housing.data, housing.target

# Regression framing: predict exact price
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cont, test_size=0.2, random_state=42
)
reg = LinearRegression().fit(X_train, y_train)
r2 = r2_score(y_test, reg.predict(X_test))

# Classification framing: high-value (above median) or not
threshold = y_cont.median()
y_bin = (y_cont > threshold).astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y_bin, test_size=0.2, random_state=42)
clf = LogisticRegression(max_iter=1000, random_state=42).fit(Xtr, ytr)
auc = roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])

fig = make_subplots(rows=1, cols=2, subplot_titles=("Regression: R²", "Classification: AUC"))
fig.add_trace(go.Bar(x=["Linear Regression"], y=[r2], marker_color=PALETTE["primary"],
                      text=[f"{r2:.3f}"], textposition="outside", showlegend=False),
              row=1, col=1)
fig.add_trace(go.Bar(x=["Logistic Regression"], y=[auc], marker_color=PALETTE["secondary"],
                      text=[f"{auc:.3f}"], textposition="outside", showlegend=False),
              row=1, col=2)

# base_layout() is built for a single axis pair, not a multi-panel subplot figure,
# so its font/color values are applied here explicitly to keep both panels consistent.
fig.update_layout(
    title=dict(text="Same Data, Two Framings — California Housing",
               font=dict(size=FONT["size_title"], family=FONT["family"])),
    font=dict(family=FONT["family"]),
    paper_bgcolor=PALETTE["background"],
    plot_bgcolor=PALETTE["surface"],
    showlegend=False,
    margin=dict(l=60, r=30, t=60, b=90),
)
fig.update_xaxes(tickfont=dict(size=FONT["size_tick"]), gridcolor="#DEE2E6")
fig.update_yaxes(tickfont=dict(size=FONT["size_tick"]), gridcolor="#DEE2E6")
fig.update_yaxes(title_text="R² (variance explained)", title_font=dict(size=FONT["size_axis"]),
                  range=[0, 1.1], row=1, col=1)
fig.update_yaxes(title_text="AUC (ranking quality)", title_font=dict(size=FONT["size_axis"]),
                  range=[0, 1.1], row=1, col=2)
fig.add_annotation(
    text="Different metrics — not directly comparable by bar height.",
    xref="paper", yref="paper", x=0.5, y=-0.28, showarrow=False,
    font=dict(size=12, color=PALETTE["muted"]),
)
fig.show()

> **Whether to frame a problem as regression or classification is primarily a business decision, not a data decision — the right framing depends on what kind of answer the decision-maker actually needs.**

---
*Next up: 16 — Neural Networks Intro, a conceptual preview of the deep learning layer above classical supervised learning*